### 분석 결과로 뉴스레터 HTML 생성

In [1]:
# 라이브러리 및 데이터 불러오기
import json
from datetime import datetime
from jinja2 import Template

# 분석 데이터 로드
with open('../data/analyzed_articles.json', 'r', encoding='utf-8') as f:
    analyzed = json.load(f)

# 예측 데이터 로드
with open('../data/predictions.json', 'r', encoding='utf-8') as f:
    predictions = json.load(f)

print(f'분석 기사: {len(analyzed)}개')
print(f'예측 생성일: {predictions["generated_at"]}')

분석 기사: 32개
예측 생성일: 2026-04-12T20:25:00.378809


In [2]:
# 뉴스레이터 데이터 구성
def build_newsletter_data(analyzed: list, predictions: dict) -> dict:
    categories = {
        'politics_economy': '정치/경제',
        'global_macro': '글로벌 매크로',
        'tech_industry': '산업/테크',
        'real_estate': '부동산',
    }

    sections = []
    for cat_key, cat_name in categories.items():
        articles = [a for a in analyzed if a.get('category') == cat_key]
        # 전체 다 포함, 중요도 순 정렬
        articles = sorted(articles, key=lambda x: x.get('importance', 0), reverse=True)
        if articles:
            sections.append({
                'category': cat_name,
                'articles': articles
            })

    urgent = [a for a in analyzed if a.get('importance', 0) >= 8]
    fused = predictions.get('fused_analysis', {})
    kronos = predictions.get('kronos_predictions', {})

    return {
        'date': datetime.now().strftime('%Y년 %m월 %d일'),
        'sections': sections,
        'urgent': urgent,
        'prediction': fused,
        'kronos': kronos,
        'weekly_summary': fused.get('weekly_summary', ''),
    }

data = build_newsletter_data(analyzed, predictions)
for s in data['sections']:
    print(f'{s["category"]}: {len(s["articles"])}개')

정치/경제: 8개
글로벌 매크로: 8개
산업/테크: 8개
부동산: 8개


In [3]:
# HTML 템플릿
TEMPLATE = '''
<!DOCTYPE html>
<html lang='ko'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>Policonomy 주간 브리핑</title>
<style>
  body { font-family: 'Apple SD Gothic Neo', sans-serif; background: #f5f5f5; margin: 0; padding: 0; }
  .container { max-width: 680px; margin: 0 auto; background: #fff; }
  .header { background: #1E3A5F; color: #fff; padding: 32px 40px; }
  .header h1 { margin: 0; font-size: 28px; letter-spacing: -0.5px; }
  .header p { margin: 8px 0 0; opacity: 0.7; font-size: 14px; }
  .section { padding: 32px 40px; border-bottom: 1px solid #eee; }
  .section-title { font-size: 13px; font-weight: 700; color: #2E6DA4; text-transform: uppercase; letter-spacing: 1px; margin: 0 0 20px; }
  .article { margin-bottom: 24px; }
  .article h3 { font-size: 16px; font-weight: 700; color: #111; margin: 0 0 8px; line-height: 1.4; }
  .article p { font-size: 14px; color: #555; line-height: 1.7; margin: 0 0 6px; }
  .article a { color: #2E6DA4; font-size: 13px; text-decoration: none; }
  .sentiment { display: inline-block; font-size: 11px; font-weight: 700; padding: 2px 8px; border-radius: 20px; margin-bottom: 8px; }
  .sentiment.positive { background: #e8f5e9; color: #2e7d32; }
  .sentiment.negative { background: #ffebee; color: #c62828; }
  .sentiment.neutral { background: #f5f5f5; color: #555; }
  .urgent { background: #fff3e0; border-left: 4px solid #ff6f00; padding: 16px 20px; margin-bottom: 16px; border-radius: 0 8px 8px 0; }
  .urgent h3 { font-size: 15px; color: #e65100; margin: 0 0 6px; }
  .urgent p { font-size: 13px; color: #555; margin: 0; }
  .prediction { background: #1E3A5F; color: #fff; padding: 32px 40px; }
  .prediction h2 { font-size: 18px; margin: 0 0 24px; }
  .pred-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 24px; }
  .pred-item { background: rgba(255,255,255,0.1); border-radius: 8px; padding: 16px; }
  .pred-item .label { font-size: 12px; opacity: 0.7; margin-bottom: 8px; }
  .pred-item .value { font-size: 20px; font-weight: 700; }
  .pred-item .confidence { font-size: 12px; opacity: 0.6; margin-top: 4px; }
  .up { color: #69f0ae; }
  .down { color: #ff5252; }
  .summary { font-size: 14px; line-height: 1.8; opacity: 0.85; border-top: 1px solid rgba(255,255,255,0.2); padding-top: 20px; margin-top: 8px; }
  .footer { padding: 24px 40px; text-align: center; font-size: 12px; color: #999; }
</style>
</head>
<body>
<div class='container'>
  <!-- 헤더 -->
  <div class='header'>
    <h1>Policonomy</h1>
    <p>{{ date }} 주간 경제 브리핑</p>
  </div>

  <!-- 긴급 이슈 -->
  {% if urgent %}
  <div class='section'>
    <div class='section-title'>긴급 이슈</div>
    {% for a in urgent %}
    <div class='urgent'>
      <h3>{{ a.title }}</h3>
      <p>{{ a.economic_impact }}</p>
    </div>
    {% endfor %}
  </div>
  {% endif %}

  <!-- 카테고리별 섹션 -->
  {% for section in sections %}
  <div class='section'>
    <div class='section-title'>{{ section.category }}</div>
    {% for a in section.articles %}
    <div class='article'>
      <span class='sentiment {{ a.sentiment }}'>
        {% if a.sentiment == 'positive' %}긍정{% elif a.sentiment == 'negative' %}부정{% else %}중립{% endif %}
      </span>
      <h3>{{ a.title }}</h3>
      <p>{{ a.economic_impact }}</p>
      <p>예상: {{ a.prediction }}</p>
      {% if a.link %}<a href='{{ a.link }}'>기사 보기</a>{% endif %}
    </div>
    {% endfor %}
  </div>
  {% endfor %}

  <!-- AI 예측 -->
  <div class='prediction'>
    <h2>AI 시장 예측</h2>
    <div class='pred-grid'>
      <div class='pred-item'>
        <div class='label'>코스피</div>
        <div class='value {% if prediction.kospi.direction == "상승" %}up{% else %}down{% endif %}'>
          {{ prediction.kospi.direction }}
        </div>
        <div class='confidence'>신뢰도 {{ prediction.kospi.confidence }}%</div>
      </div>
      <div class='pred-item'>
        <div class='label'>달러/원</div>
        <div class='value {% if prediction.usd_krw.direction == "강세" %}up{% else %}down{% endif %}'>
          {{ prediction.usd_krw.direction }}
        </div>
        <div class='confidence'>신뢰도 {{ prediction.usd_krw.confidence }}%</div>
      </div>
      <div class='pred-item'>
        <div class='label'>주목 섹터</div>
        <div class='value up'>{{ prediction.key_sector.name }}</div>
        <div class='confidence'>{{ prediction.key_sector.direction }}</div>
      </div>
      <div class='pred-item'>
        <div class='label'>전반 심리</div>
        <div class='value {% if prediction.overall_sentiment == "positive" %}up{% else %}down{% endif %}'>
          {% if prediction.overall_sentiment == 'positive' %}긍정{% else %}부정{% endif %}
        </div>
      </div>
    </div>
    <div class='summary'>{{ weekly_summary }}</div>
  </div>

  <!-- 푸터 -->
  <div class='footer'>
    Policonomy | AI 기반 경제 뉴스레터<br>
    본 뉴스레터는 AI가 생성한 분석으로 투자 권유가 아닙니다.
  </div>
</div>
</body>
</html>
'''
print('템플릿 준비 완료')

템플릿 준비 완료


In [4]:
with open('../templates/newsletter.html', 'w', encoding='utf-8') as f:
    f.write(TEMPLATE)

print('저장 완료')

저장 완료


In [10]:
# HTML 생성
def generate_newsletter(data: dict) -> str:
    template = Template(TEMPLATE)
    html = template.render(**data)
    return html

html = generate_newsletter(data)
print(f'HTML 생성 완료: {len(html):,}자')

HTML 생성 완료: 19,653자


In [11]:
# 저장 + 미리보기
# 저장
output_path = '../data/newsletter.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'저장 완료: {output_path}')
print('브라우저에서 열기:')
print(f'open {output_path}')

저장 완료: ../data/newsletter.html
브라우저에서 열기:
open ../data/newsletter.html


In [12]:
import subprocess
subprocess.run(['open', '../data/newsletter.html'])

CompletedProcess(args=['open', '../data/newsletter.html'], returncode=0)